# 02_DATA PREPARATION

In [22]:
from pathlib import Path
import pandas as pd

RAW = Path("../data/raw/training")
files_A = sorted((RAW / "training_setA").glob("*.psv"))
files_B = sorted((RAW / "training_setB").glob("*.psv"))
len(files_A), len(files_B)

(20336, 20000)

In [23]:
#lecture d'un patient
def read_patient(path, hospital):
    df = pd.read_csv(path, sep="|")
    df["patient_id"] = path.stem   # nom du fichier sans ".psv", ex. "p000001"
    df["hospital"] = hospital      # "A" ou "B"
    return df

read_patient(files_A[0], "A").head()

,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,EtCO2,BaseExcess,HCO3,...,Platelets,Age,Gender,Unit1,Unit2,HospAdmTime,ICULOS,SepsisLabel,patient_id,hospital
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,83.14,0,NaN,NaN,-0.03,1,0,p000001,A
1,97.0,95.0,NaN,98.0,75.33,NaN,19.0,NaN,NaN,NaN,...,NaN,83.14,0,NaN,NaN,-0.03,2,0,p000001,A
2,89.0,99.0,NaN,122.0,86.00,NaN,22.0,NaN,NaN,NaN,...,NaN,83.14,0,NaN,NaN,-0.03,3,0,p000001,A
3,90.0,95.0,NaN,NaN,NaN,NaN,30.0,NaN,24.0,NaN,...,NaN,83.14,0,NaN,NaN,-0.03,4,0,p000001,A
4,103.0,88.5,NaN,122.0,91.33,NaN,24.5,NaN,NaN,NaN,...,NaN,83.14,0,NaN,NaN,-0.03,5,0,p000001,A


dans un fichier on ne sait pas qui est le patient et de quel hôpital il vient (car c'est le nom du fichier qui le dit)

si on empile tous les patients on perd l'information donc on ajoute 2 colonnes : 'patient_id' et 'hospital' (pour séparer A et B)

In [ ]:
#lire tous les patients et les empiler (ou relire la sauvegarde si elle existe)
PROCESSED = Path("../data/processed")
PROCESSED.mkdir(exist_ok=True)
HOURLY = PROCESSED / "hourly.pkl"

if HOURLY.exists():
    data = pd.read_pickle(HOURLY)                    # qlq secondes
else:
    frames = [read_patient(f, "A") for f in files_A] + [read_patient(f, "B") for f in files_B]
    data = pd.concat(frames, ignore_index=True)      # 10 minutes une seule fois
    data.to_pickle(HOURLY)
data.shape

(1552210, 43)

In [25]:
#verif
print(data["patient_id"].nunique())
print(data.groupby("hospital")["patient_id"].nunique())

40336
hospital
A    20336
B    20000
Name: patient_id, dtype: int64


In [26]:
#controle si les heures se suivent
step = data.groupby("patient_id")["ICULOS"].diff()
step.value_counts(dropna=False)

ICULOS
1.0    1511874
NaN      40336
Name: count, dtype: int64

on calcule l'écart avec la ligne préceédente sur la coloen 'ICULOS'. On groupby par id pour faire le calcul patient par patient, pour ne jamais comparer  la dernière heure d'un patient avec la première heure du suivant.

In [27]:
#controle de l'étiquette 
label_step = data.groupby("patient_id")["SepsisLabel"].diff()
(label_step < 0).sum()

np.int64(0)

si un patient est étiqueté sepsis (1) il doit le rester. Ici nous n'avons pas d'erreur.

In [28]:
#verif si le fichier est complet
pd.read_pickle(PROCESSED / "hourly.pkl").shape

(1552210, 43)

In [29]:
#decoupage par patient
patients = data.groupby("patient_id").agg(
    hospital=("hospital", "first"),
    n_hours=("ICULOS", "size"),
    ever_sepsis=("SepsisLabel", "max"),
).reset_index()
patients.head()

,patient_id,hospital,n_hours,ever_sepsis
0,p000001,A,54,0
1,p000002,A,23,0
2,p000003,A,48,0
3,p000004,A,29,0
4,p000005,A,48,0


In [30]:
#patients septiques
patients.groupby("hospital")["ever_sepsis"].agg(["size", "sum", "mean"])

,size,sum,mean
hospital,,,
A,20336,1790,0.088021
B,20000,1142,0.057100


In [31]:
#découpe train/test
from sklearn.model_selection import train_test_split

SEED = 42

def split_hospital(p):
    train, rest = train_test_split(p, train_size=0.6, stratify=p["ever_sepsis"], random_state=SEED)
    calib, test = train_test_split(rest, train_size=0.5, stratify=rest["ever_sepsis"], random_state=SEED)
    return pd.concat([train.assign(split="train"), calib.assign(split="calib"), test.assign(split="test")])

patients = pd.concat([split_hospital(p) for _, p in patients.groupby("hospital")], ignore_index=True)
patients.groupby(["hospital", "split"])["ever_sepsis"].agg(["size", "mean"])

size      mean
hospital split                 
A        calib   4067  0.088026
         test    4068  0.088004
         train  12201  0.088026
B        calib   4000  0.057250
         test    4000  0.057000
         train  12000  0.057083

on coupe 2 fois :
- 60% pour le train
- 40% pour le reste avec 20% pour la calibration et 20% pour le test

on découpe les 2 hôpitaux pour pouvoir faire l'expérience dans les 2 sens

résultats :
- 8.8% des patients A développent sepsis
- 5.7% pour B

In [32]:
#verif + sauvegarde
print(patients["patient_id"].is_unique)
patients.to_csv(PROCESSED / "patients_splits.csv", index=False)

True


### VARIABLES DE LA BASELINE

principe : à l'heure t, le modèle n'a le droit d'utiliser que ce qu'on savait à l'heure t. Pour les vides, on prend la dernière valeur connue du même patient (et pas celle d'un autre ou d'une valeur future)

In [33]:
#liste des variables
VARIABLES = [c for c in data.columns if c not in ["SepsisLabel", "patient_id", "hospital"]]
print(len(VARIABLES))
print(data[["Age", "Gender", "ICULOS"]].isna().sum())

40
Age       0
Gender    0
ICULOS    0
dtype: int64


In [34]:
#on remplit avec la dernière valeur connue
locf = data.groupby("patient_id")[VARIABLES].ffill() #remplace chaque case vide par la derniere val non-vide (méthode LOCF)
pd.DataFrame({"HR_brut": data["HR"], "HR_rempli": locf["HR"]}).head(8)

,HR_brut,HR_rempli
0,NaN,NaN
1,97.0,97.0
2,89.0,89.0
3,90.0,90.0
4,103.0,103.0
5,110.0,110.0
6,108.0,108.0
7,106.0,106.0


In [35]:
#indicateurs jamais mesuré
to_flag = [v for v in VARIABLES if v not in ["Age", "Gender", "ICULOS"]]
missing = locf[to_flag].isna().astype(int).add_suffix("_missing")
missing.mean().sort_values(ascending=False).head(10)

Bilirubin_direct_missing    0.953425
EtCO2_missing               0.915879
Fibrinogen_missing          0.892848
TroponinI_missing           0.849468
Lactate_missing             0.701823
SaO2_missing                0.698076
Bilirubin_total_missing     0.697902
Alkalinephos_missing        0.696910
AST_missing                 0.694009
BaseExcess_missing          0.681044
dtype: float64

Variables le plus souvent jamais mesurées dans le tableau ci-dessus.
apres remplissage, une case vide = variable jamais mesuré chez le patient jusqu'à cette heure


### TEST ANTI-FUITE

on prend un patient à l'heure t, on efface toutes ses heures apres t et on recalcule ses variables. si c'est identiques alors c'est la preuve qu'elles n'utilisent pas le futur

In [36]:
#calcul des variables
def build_features(df):
    locf = df.groupby("patient_id")[VARIABLES].ffill()
    missing = locf[to_flag].isna().astype(int).add_suffix("_missing")
    return pd.concat([df[["patient_id", "hospital", "SepsisLabel"]], locf, missing], axis=1)

pd.testing.assert_frame_equal(build_features(data), features) #compare 2 tableau si c'est c'est différent = erreur
print("La fonction redonne exactement le tableau des cellules 14-16")

La fonction redonne exactement le tableau des cellules 14-16


In [37]:
#test sur 50 patients au hasard
import numpy as np
rng = np.random.default_rng(SEED)

for pid in rng.choice(data["patient_id"].unique(), size=50, replace=False):
    one = data[data["patient_id"] == pid]            # le séjour complet du patient
    cut = rng.integers(1, len(one) + 1)              # on garde les `cut` premières heures
    truncated = one.iloc[:cut]                       # ce qu'on savait à ce moment-là
    pd.testing.assert_frame_equal(build_features(truncated), features.loc[truncated.index])

print("Test anti-fuite réussi : sur 50 patients, aucune variable n'utilise le futur")

Test anti-fuite réussi : sur 50 patients, aucune variable n'utilise le futur


In [ ]:
#verif si le test détecte une fuite
#on lance sur une version fausse (qui remplit les trous avec la prochaine val 'bfill')
def build_features_leaky(df):
    return df.groupby("patient_id")[VARIABLES].bfill()   # FAUX exprès : utilise le futur

n_detected = 0
for pid in rng.choice(data["patient_id"].unique(), size=50, replace=False):
    one = data[data["patient_id"] == pid]
    cut = rng.integers(1, len(one) + 1)
    if not build_features_leaky(one.iloc[:cut]).equals(build_features_leaky(one).iloc[:cut]): #renvoie T ou F au lieu d'une erreur
        n_detected += 1

print(f"Fuite détectée chez {n_detected} patients sur 50")

Fuite détectée chez 43 patients sur 50
